In [2]:
# Cell 1: Install & Configure with Mistral

!pip install pypdf opensearch-py requests-aws4auth boto3 pydantic rich python-dotenv

import os
import sys
import json
import boto3
from pathlib import Path
from rich.console import Console
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TaskProgressColumn
import time

# Clear AWS_PROFILE (SageMaker uses IAM role, not profiles)
if 'AWS_PROFILE' in os.environ:
    del os.environ['AWS_PROFILE']

# Set AWS mode
os.environ['USE_AWS'] = 'true'
os.environ['AWS_REGION'] = 'us-east-1'

# Get OpenSearch endpoint dynamically


os.environ['OPENSEARCH_ENDPOINT'] = 'https://search-ms-sos-legal-vector-awmjbt2db4a4gn4jhml5zpurm4.us-east-1.es.amazonaws.com'
os.environ['OPENSEARCH_INDEX'] = 'ms-legal-abstracts'
os.environ['S3_BUCKET'] = 'ms-sos-legal-documents'

# 🎯 USE MISTRAL 7B (No Marketplace permissions required!)
os.environ['BEDROCK_CLAUDE_MODEL'] = 'anthropic.claude-haiku-4-5-20251001-v1:0'
os.environ['BEDROCK_EMBEDDING_MODEL'] = 'amazon.titan-embed-text-v1'

console = Console()
console.print("[bold green]✅ Environment configured with Mistral 7B![/bold green]")
console.print(f"[cyan]OpenSearch domain: ms-sos-legal-vector[/cyan]")
console.print(f"[cyan]Endpoint: {os.environ['OPENSEARCH_ENDPOINT']}[/cyan]")
console.print(f"[cyan]Model: {os.environ['BEDROCK_CLAUDE_MODEL']}[/cyan]")
console.print(f"[cyan]S3 Bucket: {os.environ['S3_BUCKET']}[/cyan]")

  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 204.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [opensearch-py]0m [opensearch-py]


✅ Environment configured with Mistral 7B!

OpenSearch domain: ms-sos-legal-vector

Endpoint: https://search-ms-sos-legal-vector-awmjbt2db4a4gn4jhml5zpurm4.us-east-1.es.amazonaws.com

Model: anthropic.claude-haiku-4-5-20251001-v1:0

S3 Bucket: ms-sos-legal-documents

In [3]:
# Cell 2: Import project modules

import sys
sys.path.insert(0, '/home/sagemaker-user')

# Force reload to pick up any code changes
import importlib
for module in ['config', 'models', 'compression_agent_bedrock', 'vector_store_opensearch', 'ingest']:
    if module in sys.modules:
        importlib.reload(sys.modules[module])

from config import config
from models import CompressedAbstract

console.print("[bold green]✅ All modules loaded successfully![/bold green]")
console.print(f"[cyan]Using model: {config.aws.bedrock_claude_model}[/cyan]")

✅ All modules loaded successfully!

Using model: anthropic.claude-haiku-4-5-20251001-v1:0

In [4]:
# Cell 3: Verify Bedrock and OpenSearch connections

console.print("[bold blue]🔍 Verifying AWS Connections...[/bold blue]\n")

# Test 1: Bedrock Mistral
console.print("[cyan]Test 1: Bedrock Mistral Access[/cyan]")
try:
    bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')
    response = bedrock_runtime.invoke_model(
        modelId='mistral.ministral-3-14b-instruct',
        body=json.dumps({
            "prompt": "<s>[INST] Say 'Ready!' [/INST]",
            "max_tokens": 10,
            "temperature": 0.7
        })
    )
    result = json.loads(response['body'].read())
    console.print(f"[green]   ✅ Bedrock Mistral: {result['outputs'][0]['text'].strip()}[/green]\n")
except Exception as e:
    console.print(f"[red]   ❌ Bedrock Error: {e}[/red]\n")

# Test 2: OpenSearch
console.print("[cyan]Test 2: OpenSearch Access[/cyan]")
try:
    vector_store = OpenSearchVectorStore()
    vector_store._create_index_if_not_exists()
    console.print("[green]   ✅ OpenSearch connected![/green]\n")
except Exception as e:
    console.print(f"[red]   ❌ OpenSearch Error: {e}[/red]\n")

# Test 3: S3
console.print("[cyan]Test 3: S3 Access[/cyan]")
try:
    s3 = boto3.client('s3')
    response = s3.list_objects_v2(Bucket='ms-sos-legal-documents', MaxKeys=5)
    count = response.get('KeyCount', 0)
    console.print(f"[green]   ✅ S3 accessible ({count} objects listed)[/green]\n")
except Exception as e:
    console.print(f"[red]   ❌ S3 Error: {e}[/red]\n")

console.print("[bold green]✅ All connections verified![/bold green]")

🔍 Verifying AWS Connections...

Test 1: Bedrock Mistral Access

   ❌ Bedrock Error: An error occurred (ValidationException) when calling the InvokeModel operation: 
{"error":{"code":"validation_error","message":"Failed to deserialize the JSON body into the target type: missing 
field `messages` at line 1 column 135","param":null,"type":"invalid_request_error"}}

Test 2: OpenSearch Access

   ❌ OpenSearch Error: name 'OpenSearchVectorStore' is not defined

Test 3: S3 Access

   ✅ S3 accessible (5 objects listed)

✅ All connections verified!

In [4]:
# Cell 4: Initialize ingestion components

console.print("[bold blue]🔧 Initializing Components...[/bold blue]\n")

# Import ingest components
from ingest import DocumentLoader

# Initialize AWS components (imported by ingest.py based on config.aws)
from compression_agent_bedrock import BedrockCompressionAgent as CompressionAgent
from vector_store_opensearch import OpenSearchVectorStore as VectorStore

# Initialize vector store
vector_store = VectorStore()
console.print("[green]✅ OpenSearch vector store initialized[/green]")

# Initialize Mistral compression agent
compression_agent = CompressionAgent()
console.print("[green]✅ Mistral compression agent initialized[/green]")

# Initialize document loader (from ingest.py)
document_loader = DocumentLoader(
    chunk_size=config.documents.chunk_size,
    chunk_overlap=config.documents.chunk_overlap
)
console.print("[green]✅ Document loader initialized[/green]")

console.print("\n[bold green]✅ All components ready![/bold green]")

🔧 Initializing Components...

✅ OpenSearch vector store initialized

✅ Mistral compression agent initialized

✅ Document loader initialized

✅ All components ready!

In [5]:
# Cell 5: Process documents with Mistral

import tempfile
from pathlib import Path

console.print("[bold blue]📄 Processing Legal Documents with Mistral 7B...[/bold blue]\n")

# Get all PDFs from S3
s3 = boto3.client('s3')
response = s3.list_objects_v2(Bucket='ms-sos-legal-documents')
pdf_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.pdf')]

# TEST WITH 5 DOCUMENTS FIRST
test_files = pdf_files[:20]
console.print(f"[yellow]⚠️  Testing with {len(test_files)} documents first[/yellow]\n")

processed_count = 0
failed_count = 0

with Progress(
    SpinnerColumn(),
    TextColumn("[progress.description]{task.description}"),
    BarColumn(),
    TaskProgressColumn(),
    console=console
) as progress:
    
    task = progress.add_task("Processing...", total=len(test_files))
    
    for pdf_key in test_files:
        try:
            progress.update(task, description=f"{processed_count + 1}/{len(test_files)}: {pdf_key}")
            
            # 1. Download PDF from S3 to temp file
            with tempfile.NamedTemporaryFile(suffix='.pdf', delete=False) as tmp_file:
                s3.download_file('ms-sos-legal-documents', pdf_key, tmp_file.name)
                tmp_path = Path(tmp_file.name)
            
            # 2. Load and chunk PDF using DocumentLoader (from ingest.py)
            chunks = list(document_loader.load_pdf(tmp_path))
            console.print(f"  {pdf_key}: {len(chunks)} chunks")
            
            # 3. Compress chunks with Mistral
            abstracts = compression_agent.compress_batch(chunks)
            
            # 4. Store in OpenSearch
            vector_store.add_abstracts(abstracts)
            
            # Clean up temp file
            tmp_path.unlink()
            
            console.print(f"  [green]✅ {pdf_key} completed ({len(abstracts)} abstracts)[/green]")
            processed_count += 1
            
        except Exception as e:
            console.print(f"  [red]❌ {pdf_key}: {e}[/red]")
            failed_count += 1
        
        progress.advance(task)

console.print(f"\n[bold green]✅ Processing complete![/bold green]")
console.print(f"[cyan]Processed: {processed_count}/{len(test_files)}[/cyan]")
console.print(f"[cyan]Failed: {failed_count}[/cyan]")

if processed_count == len(test_files):
    console.print("\n[bold green]🎉 All test documents processed successfully![/bold green]")
    console.print("[yellow]Ready to process all documents? Update test_files = pdf_files[:]</yellow>")

📄 Processing Legal Documents with Mistral 7B...

⚠️  Testing with 20 documents first

Output()

incorrect startxref pointer(3)

parsing for Object Streams

...

source-documents/00000066c.pdf: 25 chunks

❌ source-documents/00000066c.pdf: AuthorizationException(403, '')

✅ Processing complete!

Processed: 4/20

Failed: 16

In [16]:
import boto3
import json

from botocore.exceptions import ClientError

# Create a Bedrock Runtime client in the AWS Region of your choice.
client = boto3.client("bedrock-runtime", region_name="us-west-2")

# Set the model ID, e.g., Llama 3 70b Instruct.
model_id = "meta.llama3-70b-instruct-v1:0"

# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Embed the prompt in Llama 3's instruction format.
formatted_prompt = f"""
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{prompt}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

# Format the request payload using the model's native structure.
native_request = {
    "prompt": formatted_prompt,
    "max_gen_len": 512,
    "temperature": 0.5,
}

# Convert the native request to JSON.
request = json.dumps(native_request)

try:
    # Invoke the model with the request.
    response = client.invoke_model(modelId=model_id, body=request)

except (ClientError, Exception) as e:
    print(f"ERROR: Can't invoke '{model_id}'. Reason: {e}")
    exit(1)

# Decode the response body.
model_response = json.loads(response["body"].read())

# Extract and print the response text.
response_text = model_response["generation"]
print(response_text)

ERROR: Can't invoke 'meta.llama3-70b-instruct-v1:0'. Reason: An error occurred (AccessDeniedException) when calling the InvokeModel operation: User: arn:aws:sts::282027772524:assumed-role/Sagemaker_Role/SageMaker is not authorized to perform: bedrock:InvokeModel on resource: arn:aws:bedrock:us-west-2::foundation-model/meta.llama3-70b-instruct-v1:0 with an explicit deny in a service control policy: arn:aws:organizations::401042946100:policy/o-p1hakak5mo/service_control_policy/p-bzauo53j


KeyError: 'body'